# Supports, Boundary Conditions & Load Cases

**Notebook 02** — Part 3 of the Tuba v4 course.

---

### What you'll learn

- All **5 support types**: anchor, guide, rest, spring, custom
- Custom DOF blocking for precise boundary conditions
- Spring stiffness matrices for directional elastic restraints
- Concentrated masses and friction coefficients
- Defining **load cases** for operating, cold, and hydrotest conditions

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pyvista as pv
from tuba import Model, Material, PipeSection, PipingBuilder
from tuba.visualizer import plots
from tuba.visualizer.pipeline import build_mesh_from_model, inflate_tubes

# Defaults to zoomable 'client' locally; set TUBA_NOTEBOOK_BACKEND=static for nbconvert/CI.
from tuba.visualizer.notebook import configure_notebook_backend
JUPYTER_BACKEND = configure_notebook_backend()
print("Imports ready.")

---

## 1 — Support Types Overview

Tuba v4 provides five built-in support types. Each maps to a distinct structural behaviour
and renders as a unique 3D shape in the visualiser.

| Type | Structural Behaviour | 3D Shape | Colour |
|:---------|:------------------------------------------------|:--------------------------------------|:----------|
| **anchor** | All 6 DOFs fixed (fully restrained) | Steel flange block | 🔴 Red |
| **guide** | Lateral restraint — allows axial sliding | Collar cylinder + bumper plates | 🟢 Green |
| **rest** | Vertical support — allows horizontal sliding | Rib plate + sliding base plate | 🔵 Blue |
| **spring** | Elastic restraint with specified stiffness | Canister + rod + clamp torus | 🟡 Yellow |
| **custom** | User-defined blocked DOFs | Sphere (generic fallback) | 🟣 Magenta |

> **Tip:** Choosing the right support type is critical for realistic stress results.
> Over-constraining a system (too many anchors) artificially raises thermal expansion stresses.

In [ ]:
# ── Build a model showcasing every support type ──────────────────────

model = Model("SupportDemo", standard="ASME_B31.3")

# Material: Carbon Steel
model.add_material(
    "Steel",
    E=2.1e11,       # Young's modulus [Pa]
    nu=0.3,         # Poisson's ratio
    rho=7850.0,     # Density [kg/m³]
    alpha=1.2e-5,   # Thermal expansion [1/°C]
    allowable_stress={20.0: 137e6, 100.0: 130e6, 200.0: 120e6},
)

# Pipe section: DN100 (4" Sch 40)
model.add_pipe_section("DN100", OD=0.1143, WT=0.00602, corrosion_allowance=0.001)

# ── Piping route: 10 m straight run along +X ────────────────────────
with model.pipe(section="DN100", material="Steel") as b:
    b.start([0, 0, 0], support="anchor")          # Node 0 — ANCHOR (red)
    b.run(2.0)
    b.add_support(type="guide")                    # @ 2 m  — GUIDE (green)
    b.run(2.0)
    b.add_support(type="rest")                     # @ 4 m  — REST (blue)
    b.run(2.0)
    b.spring(y=1.5e6)                                # @ 6 m  — Y-SPRING (yellow)
    b.run(2.0)
    b.add_support(type="custom",                   # @ 8 m  — CUSTOM (magenta)
                  blocked_dof=[1, 1, 0, 0, 0, 1])  # Tx, Ty, Rz blocked
    b.run(2.0)
    b.end(support="anchor")                        # Node end — ANCHOR (red)

# ── Inspect supports ────────────────────────────────────────────────
print(f"Model '{model.project_name}' — {len(model.supports)} supports\n")
for s in model.supports:
    print(f"  Node {s.node:>4s}  │  type={s.type:<8s}  │  props: "
          f"stiffness_matrix={getattr(s, 'stiffness_matrix', '—')}  "
          f"blocked_dof={getattr(s, 'blocked_dof', '—')}")

In [ ]:
# ── 3D Render: every support shape in context ───────────────────────

mesh = build_mesh_from_model(model)
tubes = inflate_tubes(mesh, radius=0.05)

p = pv.Plotter()
p.set_background("#1a1a2e")

# Add pipe tubes
p.add_mesh(tubes, color="#c0c0c0", smooth_shading=True, opacity=0.85)

# Add 3D support shapes (anchor blocks, guide collars, etc.)
plots._add_supports_to_plotter(p, model, scale=0.18)

# Legend
p.add_legend(
    [
        ["Anchor", "red"],
        ["Guide", "green"],
        ["Rest", "blue"],
        ["Spring", "yellow"],
        ["Custom", "magenta"],
    ],
    bcolor="#2a2a3e",
    face="circle",
)

p.camera_position = "xz"
p.show(jupyter_backend=JUPYTER_BACKEND)

---

## 2 — Advanced Support Configuration

Beyond the basic type, Tuba supports let you fine-tune:

### Spring Stiffness Matrix

A 6-component vector `[kx, ky, kz, krx, kry, krz]` defines **directional stiffness**
for translational and rotational DOFs independently. This models real-world spring hangers
or snubbers that resist motion in specific directions only.

```
stiffness_matrix = [1e5, 2e5, 3e5, 0, 0, 0]
                    ───  ───  ───  ─  ─  ─
                     kx   ky   kz  free rotations
```

### Concentrated Mass

Attach a **lumped mass** (kg) at a support node — e.g. for a valve, flange, or instrument
cluster. This mass participates in gravity and dynamic load cases.

### Friction Coefficient

For `rest` supports, a Coulomb **friction coefficient** controls the lateral resistance
before sliding occurs. A typical value for steel-on-steel is `μ = 0.3`.
When friction is set, the solver accounts for potential lift-off under upward loads.

In [ ]:
# ── Advanced support examples ───────────────────────────────────────

# Directional spring: stiff vertically, softer laterally, free rotation
model.add_support(
    node="N3",
    type="spring",
    stiffness_matrix=[1e5, 2e5, 3e5, 0, 0, 0],
)

# Rest with concentrated mass (valve weight) and friction
model.add_support(
    node="N2",
    type="rest",
    mass=50.0,                  # 50 kg lumped mass
    friction_coefficient=0.3,   # Coulomb μ for steel-on-steel
)

# ── Print updated supports ──────────────────────────────────────────
print(f"Updated support count: {len(model.supports)}\n")
for s in model.supports:
    extras = []
    if hasattr(s, 'stiffness_matrix') and s.stiffness_matrix:
        extras.append(f"stiffness_matrix={s.stiffness_matrix}")
    if hasattr(s, 'mass') and s.mass:
        extras.append(f"mass={s.mass} kg")
    if hasattr(s, 'friction_coefficient') and s.friction_coefficient:
        extras.append(f"μ={s.friction_coefficient}")
    extra_str = ", ".join(extras) if extras else "—"
    print(f"  Node {s.node:>4s}  │  {s.type:<8s}  │  {extra_str}")

---

## 3 — Load Cases

A **load case** bundles the external actions applied to the piping system in a single
operating scenario:

| Parameter | Description |
|:------------------|:------------------------------------------------------------|
| `gravity` | Self-weight + contents weight (boolean) |
| `pressure` | Internal design pressure [Pa] |
| `temperature` | Operating temperature [°C] |
| `ref_temperature` | Installation / ambient temperature [°C] (default 20 °C) |

Thermal expansion stress is computed from `ΔT = temperature − ref_temperature`.

Real-world systems require **multiple load cases** to capture:

- **Operating Hot** — full temperature & pressure (worst-case expansion)
- **Operating Cold** — low-temperature standby
- **Hydrotest** — 1.5× design pressure at ambient temperature (code requirement)

In [ ]:
# ── Define load cases ───────────────────────────────────────────────

model.define_load_case(
    "Operating_Hot",
    gravity=True,
    pressure=2.5e6,         # 25 bar
    temperature=220.0,      # °C
    ref_temperature=20.0,   # °C  →  ΔT = 200 °C
)

model.define_load_case(
    "Operating_Cold",
    gravity=True,
    pressure=0.5e6,         # 5 bar
    temperature=50.0,       # °C
)

model.define_load_case(
    "Hydrotest",
    gravity=True,
    pressure=3.75e6,        # 1.5 × 25 bar
    temperature=20.0,       # ambient
)

# ── Inspect ──────────────────────────────────────────────────────────
print(f"{len(model.load_cases)} load cases defined:\n")
for lc in model.load_cases.values():
    print(f"  📦 {lc.name:<16s}  │  gravity={str(lc.gravity):<5s}  "
          f"│  P={lc.internal_pressure/1e6:.2f} MPa  "
          f"│  T={lc.temperature:.0f} °C  "
          f"│  T_ref={getattr(lc, 'ref_temperature', 20.0):.0f} °C")

---

## Key Takeaways

| Support | When to use |
|:---------|:------------------------------------------------------------------|
| **Anchor** | Nozzle connections, equipment tie-ins — fully fixed points |
| **Guide** | Mid-span lateral restraint while allowing thermal growth |
| **Rest** | Gravity supports on pipe racks, shoe supports |
| **Spring** | Variable or constant spring hangers for vertical flexibility |
| **Custom** | Non-standard restraints, partial fixity, directional stops |

- Use **`stiffness_matrix`** on springs when directional stiffness differs.
- Attach **`mass`** to any support node to model heavy in-line components.
- Set **`friction_coefficient`** on rests for realistic sliding behaviour.
- **Load cases** define the operating envelope — they drive the solver
  and ASME B31.3 compliance checks covered in **Notebook 03**.

---

*Next → [03 — Solving & Results](03_solving_and_results.ipynb)*